# Блокнот 15. Механика LLM своими руками

Курс «Машинное обучение с нуля», модуль 15. Разделы: 1–2 — урок 15.1, 3–5 — урок 15.3. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

Ничего не скачивается и не нужна видеокарта: мы разбираем механизмы на маленьких примерах.

In [ ]:
from collections import Counter

import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from torch import nn

rng = np.random.default_rng(0)
torch.manual_seed(0)

## 1. Токенизатор BPE с нуля (урок 15.1)

Слово — кортеж символов с меткой конца слова `_`. На каждом шаге ищем самую частую пару соседних токенов и сливаем её.

**Задание 1.** Допишите `merge_pair`: заменить в кортеже `word` каждое вхождение пары `(a, b)` одним токеном `a + b`.

In [ ]:
def merge_pair(word, pair):
    a, b = pair
    out, i = [], 0
    ...  # ваш код: пройти по word и слить пары
    return tuple(out)


assert merge_pair(("м", "о", "с", "т", "_"), ("с", "т")) == ("м", "о", "ст", "_")


def train_bpe(text, n_merges):
    vocab = {tuple(w) + ("_",): c for w, c in Counter(text.split()).items()}
    merges = []
    for _ in range(n_merges):
        pairs = Counter()
        for w, c in vocab.items():
            for p in zip(w, w[1:]):
                pairs[p] += c
        if not pairs:
            break
        best = max(pairs, key=lambda p: (pairs[p], p))
        merges.append(best)
        vocab = {merge_pair(w, best): c for w, c in vocab.items()}
    return merges


def tokenize(word, merges):
    w = tuple(word) + ("_",)
    for m in merges:
        w = merge_pair(w, m)
    return w


corpus = ("дроид чинит корабль . дроиды чинят корабли . корабль летит к звёздам . дроид летит на корабле . "
          "пилот чинит дроида . пилоты летят к звёздам . звёзды светят пилотам . маленький дроид знает корабли . ") * 3
merges = train_bpe(corpus, 60)
print("первые слияния:", merges[:8])
for w in ("пилот", "корабли", "звездолёт"):
    print(w, "→", tokenize(w, merges))

## 2. Температура и top-p (урок 15.1)

**Задание 2.** Допишите `top_p_filter`: оставить самые вероятные токены, пока их сумма не станет не меньше `p`, обнулить остальные и нормировать.

In [ ]:
words = np.array(["корабль", "дроид", "меч", "звезда", "пилот", "лес"])
logits = np.array([3.0, 2.2, 1.5, 1.0, 0.4, -0.5])


def softmax(z, T=1.0):
    e = np.exp((z - z.max()) / T)
    return e / e.sum()


def top_p_filter(probs, p):
    order = np.argsort(-probs)
    keep = ...  # ваш код: номера оставленных токенов
    out = np.zeros_like(probs)
    out[keep] = probs[keep]
    return out / out.sum()


for T in (0.5, 1.0, 2.0):
    probs = top_p_filter(softmax(logits, T), 0.9)
    draws = rng.choice(len(words), size=2000, p=probs)
    freq = Counter(words[draws])
    print(f"T = {T}: " + ", ".join(f"{w} {freq[w] / 2000:.2f}" for w in words if freq[w]))
assert np.count_nonzero(top_p_filter(softmax(logits), 0.9)) == 4

## 3. Мини-RAG (урок 15.3)

Вместо нейросетевых эмбеддингов возьмём TF-IDF — векторы, где у каждого слова вес тем больше, чем оно характернее для документа. Принцип RAG тот же: найти ближайшие куски и вставить их в промпт.

**Задание 3.** Допишите поиск: косинусная близость вопроса с каждым куском и номера `k` лучших.

In [ ]:
docs = [
    "Гипердвигатель Т-65 обслуживают каждые 200 часов полёта; допуск зазора сопла 0,3 мм.",
    "Астромеханический дроид подключается к разъёму за кабиной пилота и прокладывает курс.",
    "Дефлекторный щит заряжается 40 секунд; при перегреве щит отключается автоматически.",
    "Перед стартом пилот проверяет давление в топливных магистралях и заряд торпед.",
    "Торпеды протонные, боезапас — шесть штук; перезарядка только в ангаре.",
    "Сопла гипердвигателя чистят от нагара мягкой щёткой, без растворителей.",
    "Кабина рассчитана на одного пилота; катапульта срабатывает при отказе систем жизнеобеспечения.",
    "Дроид-механик ведёт журнал неисправностей и сообщает о них по защищённому каналу.",
]
vec = TfidfVectorizer()
D = vec.fit_transform(docs).toarray()


def retrieve(question, k=2):
    q = vec.transform([question]).toarray()[0]
    ...  # ваш код: верните номера k ближайших кусков


question = "Какой допуск зазора сопла у гипердвигателя?"
top = retrieve(question)
print("найдены куски:", top)
assert top[0] == 0
prompt = "Отвечай только по документам и указывай номер документа.\n\n" + "\n".join(
    f"[{i}] {docs[i]}" for i in top) + f"\n\nВопрос: {question}"
print(prompt)

Этот промпт и отправили бы языковой модели. В настоящих системах куски ищут по нейросетевым эмбеддингам (они понимают синонимы: «зазор» и «щель»), а базы хранят миллионы кусков.

## 4. LoRA на одном слое (урок 15.3)

Возьмём «предобученный» слой с весами $W$ и «новую задачу», где правильные веса — $W + \Delta$, а $\Delta$ имеет ранг 4. Заморозим $W$ и выучим поправку $BA$.

**Задание 4.** Допишите `forward`: выход замороженного слоя плюс поправка `x @ A.T @ B.T`.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=4):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))     # B = 0: в начале поправка нулевая

    def forward(self, x):
        ...  # ваш код


d = 64
base = nn.Linear(d, d)
delta = torch.randn(d, 4) @ torch.randn(4, d) * 0.1            # «новая задача»: поправка ранга 4
X = torch.randn(2000, d)
Y = X @ (base.weight.detach() + delta).T + base.bias.detach()
layer = LoRALinear(base, r=4)
trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
print("обучаемых параметров:", trainable, "из", sum(p.numel() for p in layer.parameters()))
opt = torch.optim.Adam([layer.A, layer.B], lr=1e-2)
for step in range(1501):
    loss = ((layer(X) - Y) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        print(f"шаг {step}: ошибка {loss.item():.5f}")
assert trainable == 2 * 4 * d and loss.item() < 1e-3

## 5. Квантование int8 (урок 15.3)

**Задание 5.** Допишите симметричное квантование группы весов: шаг `s = max|w| / 127`, целые `q = round(w / s)`; обратно — `q * s`.

In [ ]:
def quantize_int8(w):
    s, q = ..., ...  # ваш код
    return q, s


W = rng.normal(0, 0.02, size=(256, 256)).astype(np.float32)
q, s = quantize_int8(W)
err = np.abs(q * s - W).mean() / np.abs(W).mean()
print(f"шаг {s:.6f}, средняя относительная ошибка {err:.3%}, байт: {W.nbytes} → {q.nbytes}")
assert q.dtype == np.int8 and err < 0.02

# Один выброс портит всю группу — поэтому квантуют маленькими группами
W_out = W.copy()
W_out[0, 0] = 1.0
q2, s2 = quantize_int8(W_out)
print(f"с одним выбросом: ошибка {np.abs(q2 * s2 - W_out).mean() / np.abs(W_out).mean():.1%}")
groups = W_out.reshape(-1, 64)
deq = np.concatenate([(lambda q_, s_: q_ * s_)(*quantize_int8(g)) for g in groups]).reshape(W_out.shape)
print(f"группами по 64 веса: ошибка {np.abs(deq - W_out).mean() / np.abs(W_out).mean():.1%}")